# K2-v3 Reasoning and Tool Parser Smoke Test

This notebook calls a running SGLang OpenAI-compatible server and displays the parsed assistant messages.

Expected server setup for this branch:

```bash
python -m sglang.launch_server ... --reasoning-parser k2_v3 --tool-call-parser multi_format
```

The parsed fields to inspect are:
- `choices[0].message.reasoning_content`
- `choices[0].message.content`
- `choices[0].message.tool_calls`


In [ ]:
import json
import os
from copy import deepcopy

import requests
from IPython.display import JSON, Markdown, display

BASE_URL = os.environ.get("SGLANG_BASE_URL", "http://fs-mbz-gpu-818:30000/v1").rstrip("/")
ROOT_URL = BASE_URL[:-3] if BASE_URL.endswith("/v1") else BASE_URL

# Change these in one place while experimenting.
REASONING_EFFORT = os.environ.get("K2V3_REASONING_EFFORT", "medium")  # high, medium, low
TOOL_FORMAT = os.environ.get("K2V3_TOOL_FORMAT", "qwen3")  # qwen3, default, minimax, dsv32, glm, gptoss, python
TIMEOUT_S = int(os.environ.get("SGLANG_TIMEOUT_S", "240"))
RUN_TOOL_FORMAT_SWEEP = os.environ.get("K2V3_RUN_TOOL_FORMAT_SWEEP", "0") == "1"

print("BASE_URL:", BASE_URL)
print("REASONING_EFFORT:", REASONING_EFFORT)
print("TOOL_FORMAT:", TOOL_FORMAT)
print("RUN_TOOL_FORMAT_SWEEP:", RUN_TOOL_FORMAT_SWEEP)


## Server and Model Check

In [ ]:
def get_json(path, timeout=10):
    url = f"{BASE_URL}{path}" if path.startswith("/") else f"{BASE_URL}/{path}"
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()


models = get_json("/models")
MODEL = os.environ.get("SGLANG_MODEL") or models["data"][0]["id"]

display(Markdown(f"**Using model:** `{MODEL}`"))
display(JSON(models))

# Optional SGLang-specific model info endpoint.
try:
    info = requests.get(f"{ROOT_URL}/get_model_info", timeout=10)
    if info.ok:
        display(Markdown("**/get_model_info**"))
        display(JSON(info.json()))
except Exception as exc:
    print("Skipping /get_model_info:", exc)


## Helpers

`show_response(...)` displays both a compact parser-focused view and the raw first choice.

In [ ]:
def post_chat(payload, timeout=TIMEOUT_S):
    payload = deepcopy(payload)
    payload.setdefault("model", MODEL)
    url = f"{BASE_URL}/chat/completions"
    response = requests.post(url, json=payload, timeout=timeout)
    if not response.ok:
        print("HTTP", response.status_code)
        try:
            display(JSON(response.json()))
        except Exception:
            print(response.text)
        response.raise_for_status()
    return response.json()


def parsed_message(response_json):
    choice = response_json["choices"][0]
    message = choice.get("message", {})
    return {
        "finish_reason": choice.get("finish_reason"),
        "reasoning_content": message.get("reasoning_content"),
        "content": message.get("content"),
        "tool_calls": message.get("tool_calls"),
        "usage": response_json.get("usage"),
    }


def show_response(response_json, title="Response"):
    display(Markdown(f"### {title}"))
    display(Markdown("**Parsed message fields**"))
    display(JSON(parsed_message(response_json)))
    display(Markdown("**Raw first choice**"))
    display(JSON(response_json["choices"][0]))


def stream_chat(payload, timeout=TIMEOUT_S):
    payload = deepcopy(payload)
    payload.setdefault("model", MODEL)
    payload["stream"] = True
    url = f"{BASE_URL}/chat/completions"
    response = requests.post(url, json=payload, timeout=timeout, stream=True)
    response.raise_for_status()

    events = []
    reasoning_parts = []
    content_parts = []
    tool_call_deltas = []

    for line in response.iter_lines(decode_unicode=True):
        if not line or not line.startswith("data: "):
            continue
        data = line[len("data: ") :]
        if data == "[DONE]":
            break
        chunk = json.loads(data)
        choice = chunk["choices"][0]
        delta = choice.get("delta", {})
        event = {
            "finish_reason": choice.get("finish_reason"),
            "reasoning_content": delta.get("reasoning_content"),
            "content": delta.get("content"),
            "tool_calls": delta.get("tool_calls"),
        }
        events.append(event)
        if event["reasoning_content"]:
            reasoning_parts.append(event["reasoning_content"])
        if event["content"]:
            content_parts.append(event["content"])
        if event["tool_calls"]:
            tool_call_deltas.extend(event["tool_calls"])

    return {
        "reasoning_content": "".join(reasoning_parts) or None,
        "content": "".join(content_parts) or None,
        "tool_call_deltas": tool_call_deltas or None,
        "events": events,
    }


## Reasoning Parser: Non-Streaming

This should separate hidden reasoning into `reasoning_content` and the final answer into `content`.

In [ ]:
reasoning_payload = {
    "model": MODEL,
    "messages": [
        {
            "role": "user",
            "content": "Solve this carefully, then give only the final numeric answer: if x = 17 * 23, what is x + 19?",
        }
    ],
    "temperature": 0,
    "max_tokens": 512,
    "reasoning_effort": REASONING_EFFORT,
    "separate_reasoning": True,
}

reasoning_response = post_chat(reasoning_payload)
show_response(reasoning_response, f"Reasoning non-streaming ({REASONING_EFFORT})")


## Reasoning Parser: Streaming

Streaming emits reasoning deltas separately as `delta.reasoning_content`. This cell reconstructs them.

In [ ]:
stream_payload = {
    **reasoning_payload,
    "stream_reasoning": True,
}

stream_result = stream_chat(stream_payload)
display(Markdown("### Streaming reconstructed fields"))
display(JSON({k: v for k, v in stream_result.items() if k != "events"}))
display(Markdown("### First 40 stream events"))
display(JSON(stream_result["events"][:40]))


## Tool Calling Parser

This uses the OpenAI chat-completions tool schema. The server should parse model-emitted tool syntax into `message.tool_calls`.

Change `TOOL_FORMAT` in the first cell if the chat template expects another format.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get a weather forecast for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, for example San Francisco"},
                    "state": {"type": "string", "description": "Two-letter US state code, for example CA"},
                    "days": {"type": "integer", "description": "Forecast length in days"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["city", "state", "days"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_time",
            "description": "Get the current local time for a timezone.",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {"type": "string", "description": "IANA timezone, for example America/Los_Angeles"}
                },
                "required": ["timezone"],
            },
        },
    },
]

tool_payload = {
    "model": MODEL,
    "messages": [
        {
            "role": "user",
            "content": "Use the provided tools. Call get_weather for San Francisco, CA for 3 days in fahrenheit. Do not answer directly.",
        }
    ],
    "tools": tools,
    "tool_choice": "auto",
    "parallel_tool_calls": True,
    "temperature": 0,
    "max_tokens": 768,
    "reasoning_effort": REASONING_EFFORT,
    "separate_reasoning": True,
    "chat_template_kwargs": {
        "tool_format": TOOL_FORMAT,
    },
}

tool_response = post_chat(tool_payload)
show_response(tool_response, f"Tool call non-streaming (tool_format={TOOL_FORMAT})")


## Tool Calling Parser: Streaming

Streaming tool calls arrive as `delta.tool_calls`. This is useful for checking parser state transitions.

In [ ]:
tool_stream_result = stream_chat({**tool_payload, "stream_reasoning": True})
display(Markdown("### Streaming reconstructed fields"))
display(JSON({k: v for k, v in tool_stream_result.items() if k != "events"}))
display(Markdown("### First 80 stream events"))
display(JSON(tool_stream_result["events"][:80]))


## Optional: Send Mock Tool Results Back

If the previous non-streaming response produced `message.tool_calls`, this cell sends fake tool outputs back to the model and displays the final parsed assistant message.

In [ ]:
def mock_tool_result(tool_call):
    name = tool_call["function"]["name"]
    raw_args = tool_call["function"].get("arguments") or "{}"
    try:
        args = json.loads(raw_args)
    except Exception:
        args = {"raw_arguments": raw_args}

    if name == "get_weather":
        return {
            "city": args.get("city", "San Francisco"),
            "state": args.get("state", "CA"),
            "forecast": ["68F sunny", "66F partly cloudy", "65F foggy morning"],
        }
    if name == "get_time":
        return {"timezone": args.get("timezone", "America/Los_Angeles"), "time": "09:30"}
    return {"ok": True, "name": name, "args": args}


assistant_message = tool_response["choices"][0]["message"]
tool_calls = assistant_message.get("tool_calls") or []

if not tool_calls:
    display(Markdown("No parsed tool calls were returned by the previous response."))
else:
    followup_messages = tool_payload["messages"] + [assistant_message]
    for call in tool_calls:
        followup_messages.append(
            {
                "role": "tool",
                "tool_call_id": call["id"],
                "name": call["function"]["name"],
                "content": json.dumps(mock_tool_result(call)),
            }
        )

    final_payload = {
        "model": MODEL,
        "messages": followup_messages,
        "tools": tools,
        "tool_choice": "none",
        "temperature": 0,
        "max_tokens": 512,
        "reasoning_effort": REASONING_EFFORT,
        "separate_reasoning": True,
        "chat_template_kwargs": {"tool_format": TOOL_FORMAT},
    }
    final_response = post_chat(final_payload)
    show_response(final_response, "Final answer after mock tool results")


## Quick Tool Format Sweep

Run this if you are unsure which `tool_format` your template/model is emitting. It sends the same prompt with each supported multi-format dialect and reports whether parsed `tool_calls` were returned.

In [ ]:
SUPPORTED_TOOL_FORMATS = ["qwen3", "default", "minimax", "dsv32", "glm", "gptoss", "python"]

if not RUN_TOOL_FORMAT_SWEEP:
    display(Markdown("Set `RUN_TOOL_FORMAT_SWEEP = True` or `K2V3_RUN_TOOL_FORMAT_SWEEP=1` to run this optional sweep."))
else:
    sweep_results = []
    for fmt in SUPPORTED_TOOL_FORMATS:
        payload = deepcopy(tool_payload)
        payload["chat_template_kwargs"] = {"tool_format": fmt}
        try:
            response = post_chat(payload)
            parsed = parsed_message(response)
            sweep_results.append(
                {
                    "tool_format": fmt,
                    "finish_reason": parsed["finish_reason"],
                    "num_tool_calls": len(parsed["tool_calls"] or []),
                    "content_preview": (parsed["content"] or "")[:120],
                    "reasoning_chars": len(parsed["reasoning_content"] or ""),
                }
            )
        except Exception as exc:
            sweep_results.append({"tool_format": fmt, "error": repr(exc)})

    display(JSON(sweep_results))
